# 20 · Drive an interactive debugger over a PTY (MicroVM)

`run_code` is one-shot. A **pseudo-terminal** lets you hold a live, stateful
session — here a `pdb` debugger you step through command by command.

**Key concepts**
- **`sbx.pty.create(PtySize(rows, cols), on_data=...)`** — attaches a real TTY
  (openpty + a shell) and returns a `PtyHandle`: stream via `on_data`,
  `send_stdin(bytes)`, `resize(...)`, `kill()`.
- **Stateful interaction** — the process keeps running between your writes, so
  REPLs, debuggers, and prompts work as they do in a terminal.
- Needs the **`microvm`** backend (a real TTY needs a full OS).

**Prerequisites**
- AWS creds + region; image ARN in `RAITBOX_MICROVM_IMAGE`.

In [ ]:
%pip install -q raitbox

### Configuration

In [ ]:
import os
import time

from raitbox import PtySize, Sandbox

IMAGE_ARN = os.environ["RAITBOX_MICROVM_IMAGE"]

## 1 · A script worth stepping through

In [ ]:
SCRIPT = """def compute(n):
    total = 0
    for i in range(n):
        total += i * i
    return total


if __name__ == "__main__":
    print("result", compute(5))
"""

sbx = Sandbox.create(backend="microvm", template=IMAGE_ARN)
sbx.files.write("/work/prog.py", SCRIPT)

## 2 · Open a PTY running pdb
We collect output as it streams, then feed debugger commands one at a time.

In [ ]:
buffer = bytearray()
pty = sbx.pty.create(PtySize(rows=30, cols=100), on_data=buffer.extend)


def send(line, pause=0.6):
    pty.send_stdin((line + "\n").encode())
    time.sleep(pause)  # let pdb respond before the next command


send("cd /work && python -m pdb prog.py")
send("break compute")     # set a breakpoint
send("continue")          # run to it
send("args")              # show arguments

## 3 · Inspect state, then let it finish

In [ ]:
send("next")
send("print(total)")      # a live variable inside the frame
send("continue")          # run to completion
send("quit")

time.sleep(0.5)
print(bytes(buffer).decode(errors="replace")[-1200:])

In [ ]:
pty.kill()
sbx.kill()
print("done")

## Recap
- The PTY held a single `pdb` process across many writes — impossible with
  one-shot `run_code`.
- `resize()` and `on_data` streaming make this suitable for a web terminal too
  (see `get_terminal_link` in `docs/CONCEPTS.md`).

**Related:** `12-background-and-snapshots` (background commands), Human access docs.